# SVCM — ITI-97 — Expansion ValueSet & Lookup CodeSystem

**Profil** : IHE ITI SVCM
**Acteur testé** : SVCM-Terminology Consumer
**Transaction** : SVCM-ITI-97
**Référence Gazelle** : https://interop.esante.gouv.fr/tm/testing/testsDefinition/viewTestPage.seam?id=12208&cid=48550

Deux scénarios :
1. `$expand` sur un ValueSet SNOMED (structures anatomiques, code `302553009`)
2. `$lookup` POST + `$validate-code` + `$subsumes` sur CIM-11 (code `04`)

## Configuration

In [ ]:
import requests
import json
import time
import os
from datetime import datetime
from urllib.parse import quote

FHIR_BASE = "https://smt.esante.gouv.fr/fhir"
API_BASE  = "https://smt.esante.gouv.fr/api"
WP_BASE   = "https://smt.esante.gouv.fr/wp-json/ans"
API_KEY   = "ZiaoDxF8Je0CfBNzu..."   # Remplacer par la clé API complète

HTTP_RETRIES = 3
HTTP_BACKOFF = 2

HEADERS_JSON = {
    "Accept": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
HEADERS_POST = {
    "Accept": "application/fhir+json",
    "Content-Type": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
HEADERS_API = {
    "accept": "*/*",
    "X-API-KEY": API_KEY,
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}

# ── run output directory ──────────────────────────────────────────────────────
NOTEBOOK_ID = "04_iti97_expand"
RUN_TS  = datetime.now().strftime("%Y%m%dT%H%M%S")
RUN_DIR = os.path.join("runs", f"{NOTEBOOK_ID}_{RUN_TS}")
os.makedirs(RUN_DIR, exist_ok=True)

# ── helpers ───────────────────────────────────────────────────────────────────

def http_get(url, headers=None):
    if headers is None:
        headers = HEADERS_JSON
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → GET {url}")
            r = requests.get(url, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def http_post(url, body=None, headers=None):
    if headers is None:
        headers = HEADERS_POST
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → POST {url}")
            r = requests.post(url, json=body, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def save_artifact(step, filename, data, binary=False):
    """Save a test artifact to the run directory, prefixed with the step number."""
    path = os.path.join(RUN_DIR, f"step{step:03d}_{filename}")
    if binary:
        with open(path, "wb") as f:
            f.write(data)
    else:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  ✓ {path}")
    return path

def print_keys(data, *keys):
    subset = {k: data.get(k) for k in keys if k in data}
    print(json.dumps(subset, indent=2, ensure_ascii=False))

print(f"Configuration OK — sortie dans : {RUN_DIR}")


---
## Étapes 0–30 — Expansion d'un ValueSet SNOMED (`$expand`)

**Requête** : `GET /fhir/ValueSet/jdv-localisation-anatomique-cisis/$expand?code=302553009&_format=json`

In [ ]:
# Étape 0  — Construction de la requête
# Étape 10 — TRANSACTION : GET $expand
url_expand = f"{FHIR_BASE}/ValueSet/jdv-localisation-anatomique-cisis/$expand?code=302553009&_format=json"

r_expand = http_get(url_expand)
vs_expanded = r_expand.json()
save_artifact(30, "expand-result.json", vs_expanded)

# Étape 20 — Réception et intégration
expansion = vs_expanded.get("expansion", {})
contains  = expansion.get("contains", [])
total     = expansion.get("total", len(contains))

# Étape 30 — PREUVE
print("[PREUVE étape 30] ValueSet expansé :")
print_keys(vs_expanded, "resourceType", "id", "url", "version", "status")
print(f"\nExpansion total      : {total}")
print(f"Concepts retournés   : {len(contains)}")
if contains:
    print("\nPremiers concepts :")
    for c in contains[:10]:
        print(f"  {c.get('system')} | {c.get('code')} — {c.get('display')}")
    if len(contains) > 10:
        print(f"  ... ({len(contains) - 10} autres)")


---
## Étapes 40–70 — Lookup, validate-code et subsumes sur CIM-11

**Requêtes** :
- `POST /fhir/CodeSystem/$lookup` — code `04` en CIM-11
- `GET /fhir/CodeSystem/$validate-code` — validation du code `04`
- `GET /fhir/CodeSystem/$subsumes` — relations de subsomption

In [ ]:
# Étape 40 — Construction de la requête
# Étape 50 — TRANSACTION : POST $lookup code 04 en CIM-11
CIM11_SYSTEM = "https://smt.esante.gouv.fr/terminologie-cim11-mms"

lookup_body = {
    "resourceType": "Parameters",
    "parameter": [
        {"name": "system", "valueUri":  CIM11_SYSTEM},
        {"name": "code",   "valueCode": "04"},
    ],
}
r_lookup = http_post(f"{FHIR_BASE}/CodeSystem/$lookup?_format=json", lookup_body)
lookup_result = r_lookup.json()
save_artifact(50, "lookup-cim11-04.json", lookup_result)

params = {p["name"]: p for p in lookup_result.get("parameter", [])}
print(f"[PREUVE étape 60] $lookup code 04 :")
print(f"  display : {params.get('display', {}).get('valueString')}")


In [ ]:
# $validate-code — code 04 dans CIM-11 (GET)
url_vc = f"{FHIR_BASE}/CodeSystem/$validate-code?url={quote(CIM11_SYSTEM)}&code=04&_format=json"
r_vc = http_get(url_vc)
vc_result = r_vc.json()
save_artifact(60, "validate-cim11-04.json", vc_result)
vc_params = {p["name"]: p for p in vc_result.get("parameter", [])}
print(f"$validate-code code 04 — result : {vc_params.get('result', {}).get('valueBoolean')}")

# $subsumes — 04 vs 3B62.0Y  (attendu : not-subsumed — chapitres différents)
url_sub1 = f"{FHIR_BASE}/CodeSystem/$subsumes?system={quote(CIM11_SYSTEM)}&codeA=04&codeB=3B62.0Y&_format=json"
r_sub1 = http_get(url_sub1)
sub1_result = r_sub1.json()
save_artifact(60, "subsumes-04-vs-3B62.0Y.json", sub1_result)
sub1 = {p["name"]: p for p in sub1_result.get("parameter", [])}
print(f"$subsumes 04 vs 3B62.0Y : {sub1.get('outcome', {}).get('valueCode')}  (attendu: not-subsumed)")

# $subsumes — 04 vs 4A01.10  (attendu : subsumes)
url_sub2 = f"{FHIR_BASE}/CodeSystem/$subsumes?system={quote(CIM11_SYSTEM)}&codeA=04&codeB=4A01.10&_format=json"
r_sub2 = http_get(url_sub2)
sub2_result = r_sub2.json()
save_artifact(60, "subsumes-04-vs-4A01.10.json", sub2_result)
sub2 = {p["name"]: p for p in sub2_result.get("parameter", [])}
print(f"$subsumes 04 vs 4A01.10  : {sub2.get('outcome', {}).get('valueCode')}  (attendu: subsumes)")
